In [2]:
#Importacion de librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

In [4]:
#Crear dataframe
encoding = "latin1"
path = "../../../data/Obesidad_Infantil.csv"
df = pd.read_csv(path, encoding=encoding)
print(df)

     PESO (KG)  TALLA (MT)  CIRCUNFERENCIA DE CINTURA  EDAD  \
0        18.25        1.00                       59.0     5   
1        14.70        1.06                       49.0     5   
2        16.45        1.06                       49.5     5   
3        13.55        1.07                       46.0     5   
4        16.95        1.07                       52.0     5   
..         ...         ...                        ...   ...   
308      73.60        1.58                       99.0    11   
309      52.40        1.60                       71.5    11   
310      63.20        1.60                       80.0    11   
311      96.20        1.60                      106.0    11   
312      74.45        1.65                       90.0    11   

     ÍNDICE CINTURA/TALLA    IMC SEXO  
0                    0.59  18.25    F  
1                    0.46  13.08    F  
2                    0.47  14.64    F  
3                    0.43  11.84    F  
4                    0.49  14.80    F  
.. 

**Modelo de Regression**
Este modelo es un analisis que se realiza sobre datos numericos, que toma un conjunto de datos de entrada X (explicativas) y devuelve un conjunto de datos Y (predictiva).

In [6]:
#Debemos excluir la variable SEXO por no ser númerica
df_numeric = df.select_dtypes(include="number")
matrix_corr=df_numeric.corr() #.corr() es una funcion exclusiva de dataframes numericos
matrix_corr

,PESO (KG),TALLA (MT),CIRCUNFERENCIA DE CINTURA,EDAD,ÍNDICE CINTURA/TALLA,IMC
PESO (KG),1.000000,0.830357,0.913062,0.629323,0.511282,0.884710
TALLA (MT),0.830357,1.000000,0.631897,0.855203,0.026385,0.504175
CIRCUNFERENCIA DE CINTURA,0.913062,0.631897,1.000000,0.415179,0.786955,0.930365
EDAD,0.629323,0.855203,0.415179,1.000000,-0.139436,0.312798
ÍNDICE CINTURA/TALLA,0.511282,0.026385,0.786955,-0.139436,1.000000,0.798940
IMC,0.884710,0.504175,0.930365,0.312798,0.798940,1.000000


In [7]:
#Omitir cualquier target leakage
#En este caso se omitira la variable IMC por contener el PESO (KG)
df_clean=df_numeric.drop(columns=["IMC"])
df_clean.columns

Index(['PESO (KG)', 'TALLA (MT)', 'CIRCUNFERENCIA DE CINTURA', 'EDAD',
       'ÍNDICE CINTURA/TALLA'],
      dtype='str')

In [8]:
#2.- Seleccionar las variables X y y
#Se decide que el modelo prediga los valores del PESO (KG)
y =  df_numeric ["PESO (KG)"] #Y ES UNA SERIE 
X = df_numeric [['TALLA (MT)', 'CIRCUNFERENCIA DE CINTURA', 'EDAD', 'ÍNDICE CINTURA/TALLA']] #Omitimos IMC
print(y.shape, X.shape)

(313,) (313, 4)


In [9]:
#3.- Crear el modelo de Regresion 
model = LinearRegression()
type(model)

sklearn.linear_model._base.LinearRegression

**Division de los datos y reajuste del modelo**

Principalmente para evitar metricas confunsas asociadas a "Sobreajuste"

train_test_split principalmente me ayuda en la prevención de overfifting, se incluye en pipelines de desarrollo de deep learning.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=42, test_size=0.2)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(250, 4) (63, 4) (250,) (63,)


In [11]:
#Ajustar el modelo
model.fit(X_train, y_train)

LinearRegression()

In [13]:
b0 = model.intercept_
bx = model.coef_
print(f"b0={b0} y bx={bx}")

b0=16.04115323185988 y bx=[-2.22019592e+01  2.00758921e+00  1.13970551e-02 -1.68569616e+02]


Modelo de Regresión:
ŷ= 16.0411 - 22.2019X1 + 2.0075X2 + 0.0113X3 - 168.5696X4

In [15]:
#Realizar predicciones con datos que el modelo no conoce X_test
y_pred= model.predict(X_test)
y_pred.shape

(63,)

In [16]:
#Imprimir metricas
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(r2, mae, mse)

0.9072634783042092 2.2447850192979333 10.18732607396


train_split (youtube) y sobreajuste

**Guardar el modelo entrenado**
Se almacena en formato .joblib(estandar del scikitlearn) para ser retomado en otro notebook para analisis residual.

In [17]:
import joblib

#Empaquetar el modelo entrenado, sus metrticas y conjuntos de datos.
artifacts={
    "model": model,
    "features": list(X.columns),
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
    "y_test": y_test,
    "metrics": {
        "r2": r2,
        "mae": mae,
        "mse": mse
    }
}

#Guardar
joblib.dump(artifacts,"../../../models/exp02_ols_no_imc/multiple_regression.joblib")

['../../../models/exp02_ols_no_imc/multiple_regression.joblib']